# 교육학 연구자를 위한 Computational NLP — 손으로 해보기
**고려대학교 특강 · 참여활동 (Google Colab)**

> 설치·코딩 경험이 없어도 됩니다. 상단 메뉴 **런타임 → 모두 실행**(Runtime → Run all)만 누르면 끝.
> GPU도, API 키도, 회원가입도 필요 없습니다. (정적 임베딩 = CPU만으로 동작)

오늘 확인할 것: **컴퓨터가 '단어를 세는' 게 아니라 '의미를 좌표로' 다룬다**는 것.

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_01.png" width="600">
<sub>📊 관련 강의 슬라이드 — 고려대 Computational NLP 특강</sub>


## 0단계 · 준비 (약 20초)
임베딩 라이브러리를 설치합니다. 아래 셀의 ▶ 재생 버튼을 누르세요.


In [ ]:
# torch/GPU 불필요한 가벼운 정적 임베딩 라이브러리
!pip -q install model2vec

**🔍 코드 뜯어보기**
- 맨 앞의 `#` 줄 — 실행되지 않는 **메모(주석)**. 코드 설명용입니다.
- `!pip install model2vec` — 느낌표 `!`는 ‘파이썬이 아니라 설치 명령’이라는 뜻. 임베딩 도구를 Colab에 설치합니다(한 번만 실행하면 됨).


## 1단계 · 의미를 좌표로 바꾸기
문장을 숫자 벡터(좌표)로 바꾸는 다국어 모델을 불러옵니다.
`encode()` 한 번이면 한국어 문장이 의미 공간의 한 점이 됩니다.

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_17.png" width="600">
<sub>📊 관련 강의 슬라이드 — BERT · SBERT — 문장을 의미 좌표로 (직관)</sub>


In [ ]:
from model2vec import StaticModel
import numpy as np

# 한국어를 포함한 다국어 정적 임베딩 (약 50MB, 자동 다운로드)
model = StaticModel.from_pretrained('minishlab/potion-multilingual-128M')

def embed(texts):
    v = model.encode(texts)
    return v / np.linalg.norm(v, axis=1, keepdims=True)   # 길이 1로 정규화

def similarity(a, b):
    return float(np.dot(a, b))   # 코사인 유사도 (1에 가까울수록 비슷)

print('준비 완료 — 문장 하나가', model.encode(['안녕하세요']).shape[1], '차원 좌표가 됩니다.')

**🔍 코드 뜯어보기**
- `from ... import` / `import numpy as np` — 필요한 도구를 불러옵니다(넘파이 = 숫자 계산기).
- `StaticModel.from_pretrained(...)` — 미리 학습된 다국어 임베딩 모델을 내려받아 `model`에 담습니다.
- `def embed(texts):` — 문장을 벡터(좌표)로 바꾸는 **나만의 함수**. `encode` 뒤 길이를 1로 맞춰(정규화) 서로 비교하기 좋게 만듭니다.
- `def similarity(a, b):` — 두 좌표의 **코사인 유사도**(1에 가까울수록 비슷). `np.dot` = 내적.
- 마지막 `print(...)` — 문장 하나가 몇 차원 좌표가 되는지 확인용.


## 2단계 · 같은 단어, 다른 의미
'죽겠다'와 '죽인다'는 글자가 닮았지만 감정은 정반대입니다.
**단어를 세는 도구는 이 둘을 구분하지 못하지만, 임베딩은 어떨까요?**

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_16.png" width="600">
<sub>📊 관련 강의 슬라이드 — 같은 단어, 다른 의미 — 어휘의 한계</sub>


In [ ]:
sents = [
    '이 문제 진짜 죽겠다, 너무 어려워서 포기하고 싶어.',     # 좌절
    '와 이거 완전 죽인다, 이렇게 재밌는 실험은 처음이야.',     # 흥분
    '계속 오류가 나서 코드를 한 줄씩 다시 읽어봤다.',         # 성찰/전략
]
E = embed(sents)
print(f'좌절 ↔ 흥분 유사도 : {similarity(E[0], E[1]):.3f}')
print(f'좌절 ↔ 성찰 유사도 : {similarity(E[0], E[2]):.3f}')
print()
print('해석: 표면 글자는 닮았어도(죽겠다/죽인다) 유사도는 낮게 벌어집니다.')
print('     감정은 단어가 아니라 "맥락"에서 나오기 때문입니다.')

**🔍 코드 뜯어보기**
- `sents = [ ... ]` — 비교할 문장 3개를 목록(리스트)에 담습니다. `#` 뒤는 설명 메모.
- `E = embed(sents)` — 세 문장을 한꺼번에 좌표로. `E[0]` = 첫 문장, `E[1]` = 둘째 …
- `similarity(E[0], E[1])` — ‘좌절’과 ‘흥분’ 문장의 유사도. `:.3f` = 소수 셋째 자리까지 표시.


## 3단계 · 의미로 검색하기 (키워드가 아니라 뜻)
질문과 **같은 단어가 하나도 없어도**, 뜻이 가까우면 찾아냅니다.
아래는 '자기조절학습'을 검색어로 네 문장을 정렬합니다.

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_21.png" width="600">
<sub>📊 관련 강의 슬라이드 — 의미 검색 — 키워드가 아니라 ‘뜻’으로</sub>


In [ ]:
corpus = [
    '학생이 실수를 스스로 발견하고 방법을 바꿨다.',
    '교사가 따뜻하게 격려하며 다음 단계를 제안했다.',
    '오늘 날씨가 맑아서 소풍을 갔다.',
    '발표가 긴장됐지만 핵심 논지는 분명했다.',
]
C = embed(corpus)
query = embed(['자기 조절 학습, 스스로 전략 수정'])[0]

scores = [similarity(query, c) for c in C]
for i in np.argsort(scores)[::-1]:
    print(f'{scores[i]:+.3f}   {corpus[i]}')
print()
print('해석: "자기조절"이란 단어가 없는 첫 문장이 1위,')
print('     무관한 "소풍" 문장은 꼴찌(음수). = 키워드가 아니라 의미로 찾음.')

**🔍 코드 뜯어보기**
- `corpus` — 뒤질 문장 창고. `query` — 찾고 싶은 뜻(검색어)을 좌표로.
- `scores = [similarity(query, c) for c in C]` — 검색어와 각 문장의 유사도를 모두 계산(반복문 축약형).
- `np.argsort(scores)[::-1]` — 점수 **높은 순**으로 자리 번호를 정렬(`[::-1]` = 뒤집기) → 위에서부터 가장 비슷한 문장.


## 4단계 · LIWC식 ‘단어 세기’와 비교하기
임베딩 이전의 고전적 방법은 **감정 사전으로 단어를 세는 것**입니다(LIWC 방식).
해석은 쉽지만, 아래처럼 **부정어**에 약합니다.

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_15.png" width="600">
<sub>📊 관련 강의 슬라이드 — LIWC · 어휘 기반 — 해석 가능한 baseline</sub>


In [ ]:
neg = {'어렵','힘들','짜증','포기','실패','좌절'}   # 부정 감정 사전(축소판)
pos = {'재미','좋','신기','성공','자신','뿌듯'}   # 긍정 감정 사전

tests = [
    '이 문제 진짜 어려워서 포기하고 싶어.',
    '이 과제 하나도 안 어렵고 재미있었다.',   # 실제로는 긍정!
    '실험이 신기하고 뿌듯했다.',
]
for s in tests:
    n = sum(w in s for w in neg); p = sum(w in s for w in pos)
    verdict = '부정' if n > p else ('긍정' if p > n else '중립')
    print(f'neg={n} pos={p} → {verdict:2s} | {s}')
print()
print('해석: 2번은 "안 어렵고 재미있었다"(긍정)인데 "어렵"이 잡혀 오판 위험.')
print('     단어 세기는 부정·맥락·반어에 약합니다 → 그래서 임베딩/맥락이 필요.')

**🔍 코드 뜯어보기**
- `neg = { ... }` / `pos = { ... }` — 부정·긍정 단어 **사전**(중괄호 = 집합).
- `n = sum(w in s for w in neg)` — 문장 `s` 안에 사전 단어가 몇 개 들어있는지 셉니다.
- `verdict = '부정' if n > p else ...` — 부정 단어가 더 많으면 ‘부정’, 아니면 ‘긍정/중립’으로 판정.
- 포인트: ‘안 어렵다’의 **‘안’**(부정어)을 못 보고 ‘어렵’만 세는 한계 → 그래서 맥락 임베딩이 필요.


## 5단계 · 문장들을 2D 지도에 그려보기
임베딩(수백 차원)을 **2차원으로 눌러** 눈으로 봅니다. 가까운 점 = 비슷한 뜻.

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_20.png" width="600">
<sub>📊 관련 강의 슬라이드 — Sentence-BERT — 의미로 닮음을 잰다</sub>


In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

sents6 = [
    '학생이 스스로 오류를 찾아 방법을 바꿨다.',
    '교사가 따뜻하게 격려했다.',
    '오늘 소풍을 갔다 즐거웠다.',
    '발표가 긴장됐지만 해냈다.',
    '실험 데이터를 정리했다.',
    '친구와 점심을 먹었다.',
]
P = PCA(n_components=2).fit_transform(embed(sents6))
plt.figure(figsize=(7,5))
plt.scatter(P[:,0], P[:,1], s=420, c=range(len(sents6)), cmap='tab10')
for i,(x,y) in enumerate(P):
    plt.annotate(str(i+1), (x,y), ha='center', va='center', color='white', fontweight='bold')
plt.title('Sentences in 2D meaning space (PCA)'); plt.xlabel('dim 1'); plt.ylabel('dim 2')
plt.grid(alpha=.3); plt.show()
for i,s in enumerate(sents6): print(i+1, s)

**🔍 코드 뜯어보기**
- `PCA(n_components=2).fit_transform(...)` — 수백 차원 좌표를 그림용 **2차원**으로 압축.
- `plt.scatter(P[:,0], P[:,1], ...)` — 각 문장을 점으로 찍기. `P[:,0]` = 가로값, `P[:,1]` = 세로값.
- `plt.annotate(str(i+1), ...)` — 점 위에 번호 쓰기. 아래 `print`가 번호↔문장 대응표.
- `plt.show()` — 그림 출력. **가까운 점일수록 뜻이 비슷**.


## 6단계 · 감정은 점수가 아니라 ‘궤적’
성찰문을 문장 순서대로 두고, 각 문장이 **좌절/자신감 앵커**와 얼마나 가까운지 그립니다.
좌절이 올랐다가 회복되는 **흐름**이 보이면, 감정은 한 숫자가 아니라 과정임을 뜻합니다.

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_29.png" width="600">
<sub>📊 관련 강의 슬라이드 — 감정은 점수가 아니라 ‘궤적’이다</sub>


In [ ]:
traj = [
    '처음엔 정말 짜증나서 포기하고 싶었다.',
    '계속 오류가 나서 답답했다.',
    '한 줄씩 다시 읽어봤다.',
    '원인을 찾았다.',
    '다음엔 더 잘할 수 있겠다.',
]
a_neg = embed(['좌절 답답 포기'])[0]     # 좌절 앵커
a_pos = embed(['자신감 성취 희망'])[0]   # 자신감 앵커
Tt = embed(traj)
xs = range(1, len(traj)+1)
plt.figure(figsize=(7,4))
plt.plot(xs, [similarity(Tt[i], a_neg) for i in range(len(traj))], 'o-', label='frustration')
plt.plot(xs, [similarity(Tt[i], a_pos) for i in range(len(traj))], 's-', label='confidence')
plt.xlabel('sentence #'); plt.ylabel('similarity to anchor'); plt.title('Emotion trajectory')
plt.legend(); plt.grid(alpha=.3); plt.show()
for i,s in enumerate(traj): print(i+1, s)

**🔍 코드 뜯어보기**
- `a_neg`, `a_pos` — ‘좌절’·‘자신감’을 대표하는 **기준점(앵커)** 좌표.
- `plt.plot(xs, [similarity(Tt[i], a_neg) ...], 'o-')` — 문장 순서(가로)에 따라 좌절 앵커와의 유사도(세로)를 선으로 잇기.
- 두 선(frustration/confidence)의 **오르내림**이 곧 감정의 궤적.


## 7단계 · 라벨 없이 주제 묶기 (군집화)
질문 글들을 **정답 라벨 없이** 비슷한 것끼리 자동으로 묶습니다(BERTopic의 축소판).

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_22.png" width="600">
<sub>📊 관련 강의 슬라이드 — BERTopic — 라벨 없이 주제를 발견</sub>


In [ ]:
from sklearn.cluster import KMeans
posts = [
    '시험 범위가 어디까지인가요?',
    '과제 마감이 언제죠?',
    '개념이 이해가 안 돼요 도와주세요',
    '이 부분 원리가 궁금합니다',
    '제출은 어디에 하나요?',
    '왜 이렇게 되는지 설명해주세요',
]
labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(embed(posts))
for c in sorted(set(labels)):
    print(f'[군집 {c}]')
    for i,s in enumerate(posts):
        if labels[i]==c: print('   -', s)
print()
print('해석: 라벨을 주지 않아도 "학습 질문"과 "행정 질문"이 갈립니다.')
print('     단, 군집에 이름을 붙이는 것(=해석)은 사람의 몫입니다.')

**🔍 코드 뜯어보기**
- `KMeans(n_clusters=2, ...).fit_predict(embed(posts))` — 문장을 좌표로 바꾼 뒤 **2개 무리**로 자동 분류. 결과 `labels` = 각 문장의 무리 번호(0/1).
- 아래 반복문 — 같은 번호끼리 모아 출력.
- `n_clusters=2`를 3으로 바꾸면 무리 수가 달라집니다. 무리에 **이름 붙이기**는 사람이 합니다.


## 8단계 · 진짜 BERT ① — 맥락으로 빈칸 채우기
지금까지는 가벼운 **정적** 임베딩이었습니다. 이제 **진짜 BERT**(한국어 `klue/bert-base`)를 씁니다.
BERT는 문장의 빈칸 `[MASK]`에 **맥락에 맞는 단어**를 채웁니다. (Colab엔 transformers·torch가 이미 있음)

> 첫 실행 때 모델(~430MB)을 내려받아 20~40초 걸릴 수 있습니다. API 키는 여전히 불필요.

<img src="https://raw.githubusercontent.com/Educatian/korea-compnlp-handson/main/slides/slide_19.png" width="600">
<sub>📊 관련 강의 슬라이드 — BERT — 맥락이 의미를 만든다</sub>


In [ ]:
from transformers import pipeline
fm = pipeline('fill-mask', model='klue/bert-base')   # 첫 실행 시 다운로드
M = fm.tokenizer.mask_token   # [MASK]

for s in [f'주말에 날씨가 좋아서 {M}을 갔다.',
          f'그는 {M}를 타고 학교에 갔다.',
          f'시험에 {M}해서 정말 기뻤다.']:
    print(s)
    for p in fm(s, top_k=3):
        print(f"    {p['token_str'].strip():10s} {p['score']:.2f}")
    print()
print('해석: 같은 빈칸이라도 문장 맥락에 따라 "버스/기차"(탈것), "합격"(시험)처럼 달라집니다.')
print('     BERT는 앞뒤 단어를 함께 보고 뜻을 정합니다 = 맥락(contextual).')

**🔍 코드 뜯어보기**
- `pipeline('fill-mask', model='klue/bert-base')` — 한국어 BERT를 ‘빈칸 채우기’ 모드로 불러옵니다.
- `M = fm.tokenizer.mask_token` — 빈칸 기호 `[MASK]`를 변수에 담아 문장 `f'...{M}...'`에 끼웁니다.
- `fm(s, top_k=3)` — 빈칸에 들어갈 후보 **상위 3개**와 확신도(`score`)를 돌려줍니다.
- 같은 빈칸이라도 문장이 다르면 후보가 달라지는 것이 핵심(맥락 이해).


## 9단계 · 진짜 BERT ② — 같은 단어, 맥락 따라 다른 벡터
정적 임베딩은 ‘밤’에 **항상 같은 벡터 1개**를 줍니다. BERT는 문장마다 **새로** 만듭니다.
‘밤’(밤하늘) vs ‘밤’(군밤)을 같은 글자인데 BERT가 구분하는지 봅니다.


In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained('klue/bert-base')
bert = AutoModel.from_pretrained('klue/bert-base'); bert.eval()

def word_vector(sent, word):
    enc = tok(sent, return_offsets_mapping=True, return_tensors='pt')
    offs = enc.pop('offset_mapping')[0].tolist()
    with torch.no_grad():
        h = bert(**enc).last_hidden_state[0]         # 토큰별 맥락 벡터
    st = sent.find(word); en = st + len(word)
    idx = [i for i,(a,b) in enumerate(offs) if b > a and a < en and b > st]
    v = h[idx].mean(0).numpy(); return v / np.linalg.norm(v)
def sim(a,b): return float(np.dot(a,b))

S1 = '가을 밤 하늘에 별이 빛났다.'    # 밤 = 밤하늘
S2 = '깊은 밤에 조용히 걸었다.'        # 밤 = 밤하늘
S3 = '밤을 구워 먹으니 달콤했다.'      # 밤 = 군밤(먹는 밤)
b1, b2, b3 = word_vector(S1,'밤'), word_vector(S2,'밤'), word_vector(S3,'밤')
print(f'밤(밤하늘) ↔ 밤(밤하늘) : {sim(b1,b2):.3f}')
print(f'밤(밤하늘) ↔ 밤(군밤)   : {sim(b1,b3):.3f}')
print()
print('해석: 글자는 같은 "밤"인데, 같은 뜻(밤하늘끼리)이 더 가깝고 다른 뜻(군밤)은 멀어집니다.')
print('     BERT가 맥락으로 벡터를 새로 만들기 때문 — 정적 임베딩은 못 하는 일.')

**🔍 코드 뜯어보기** (조금 어렵지만 원리만 이해하면 됩니다)
- `tok`, `bert` — 한국어 BERT의 **토크나이저**(문장을 조각으로 쪼갬)와 **본체 모델**을 불러옵니다.
- `def word_vector(sent, word):` — 문장 속 특정 단어의 **맥락 벡터**를 뽑는 함수:
  - `bert(**enc).last_hidden_state` — 문장을 넣으면 **토큰(조각)마다** 맥락이 반영된 벡터가 나옵니다.
  - `offset`/`idx` — 그 중 우리가 원하는 단어(예: ‘밤’)에 해당하는 조각만 골라
  - `.mean(0)` — 평균 내어 그 단어 하나의 벡터로 만듭니다(정규화 포함).
- `S1·S2·S3` — 같은 글자 ‘밤’이 서로 다른 뜻(밤하늘/군밤)으로 쓰인 문장.
- 결과: 같은 뜻끼리 유사도가 높고 다른 뜻은 낮음 → BERT가 **맥락으로 벡터를 새로 만든다**는 증거.


## 10단계 · 직접 바꿔보기 ✍️
아래 문장들을 **여러분의 연구 주제**로 바꿔 다시 실행해 보세요.
(예: 학생 성찰문, 토론 글, 교사 피드백 — 단, 실제 학생 데이터 대신 예시/합성 문장으로.)


In [ ]:
my_query = '여기에 내 연구 관심을 한 문장으로'
my_texts = [
    '바꿔볼 문장 1',
    '바꿔볼 문장 2',
    '바꿔볼 문장 3',
]
Cq = embed(my_texts)
qv = embed([my_query])[0]
for i in np.argsort([similarity(qv, c) for c in Cq])[::-1]:
    print(f'{similarity(qv, Cq[i]):+.3f}   {my_texts[i]}')

**🔍 코드 뜯어보기**
- `my_query` / `my_texts` — 이 두 곳의 따옴표 안 글자만 **여러분 문장으로 바꾸면** 됩니다(나머지는 그대로).
- 아래 줄은 3단계 ‘의미 검색’과 똑같은 원리 — 검색어와 가장 가까운 문장부터 정렬해 보여줍니다.


---
## (선택) · LLM에게 공감 코딩 시키기
임베딩은 '의미의 거리'를 재고, LLM은 '라벨과 근거'를 답니다.
아래는 **본인 API 키가 있을 때만** 실행하는 선택 활동입니다. 키가 없으면 폰의 ChatGPT/Claude 앱에 같은 프롬프트를 붙여넣어도 동일합니다.
무료 키가 필요하면 **Google AI Studio**(aistudio.google.com/apikey · Gemini)에서 카드 없이 발급받을 수 있습니다(강의 슬라이드 참고).

> ⚠️ **실제 학생 데이터는 외부 모델에 넣지 마세요.** 합성/공개 예시만.


In [ ]:
PROMPT = '''다음 교사 피드백을 공감 코딩해줘. 각 문장을
[정서적 인정 / 관점 수용 / 실행 가능한 조언 / 과잉 도움] 중 하나로 분류하고,
근거를 한 줄씩 붙여줘. 분류가 애매하면 '불확실'로 표시해.

피드백: "이번 발표 준비하느라 정말 고생 많았어요. 긴장한 게 느껴졌지만
핵심 논지는 분명했어요. 다음엔 도입을 30초로 줄이고 예시를 하나만 남겨보면 어떨까요?"'''

print(PROMPT)
print()
print('↑ 이 프롬프트를 ChatGPT/Claude 앱에 붙여넣으세요.')
print('  (API 키가 있으면 아래 주석을 풀어 코드로도 호출할 수 있습니다.)')

# --- 선택 A: Google AI Studio(Gemini) 무료 키 — 강의 슬라이드 참고 ---
#   키 발급: https://aistudio.google.com/apikey  (구글 계정, 카드 없이 무료)
# !pip -q install google-genai
# from google import genai
# client = genai.Client(api_key='AIza...여기에_본인_키...')
# r = client.models.generate_content(model='gemini-2.5-flash', contents=PROMPT)
# print(r.text)

# --- 선택 B: OpenAI API 키가 있을 때 ---
# !pip -q install openai
# from openai import OpenAI
# client = OpenAI(api_key='sk-...여기에_본인_키...')
# r = client.chat.completions.create(model='gpt-4o-mini',
#         messages=[{'role':'user','content':PROMPT}])
# print(r.choices[0].message.content)

**🔍 코드 뜯어보기**
- `PROMPT = '''...'''` — 삼중 따옴표는 **여러 줄 문장**을 담는 방법. LLM에게 보낼 지시문입니다.
- `print(PROMPT)` — 그 지시문을 화면에 찍어 줍니다. 이걸 복사해 ChatGPT/Claude 앱에 붙여넣으면 끝.
- 맨 아래 `#` 줄들 — 키가 있을 때 **주석(`#`)을 지우고** 코드로 직접 호출하는 방법. 선택 A=Google Gemini(무료), 선택 B=OpenAI.


---
### 오늘 손으로 확인한 것
1. 임베딩은 문장을 **의미 좌표**로 바꾼다 (단어 세기가 아님).
2. **거리 = 의미 유사도** — 같은 단어라도 맥락이 다르면 멀어진다.
3. **LIWC식 단어 세기**는 쉽지만 부정·맥락에 약하다 → 맥락 임베딩이 보완.
4. 임베딩을 **2D로** 보면 문장 간 관계가 눈에 보인다.
5. 감정은 한 점수가 아니라 **궤적**(오르내림)으로 본다.
6. **군집화**는 라벨 없이 주제를 묶지만, 이름 붙이기(해석)는 사람의 몫.
7. **진짜 BERT**는 맥락으로 빈칸을 채우고, 같은 단어도 문맥마다 **다른 벡터**로 만든다 — 정적 임베딩은 못 하는 일.
8. LLM은 **라벨과 근거의 초안**을 준다 — 신뢰도·과잉주장 검증은 연구자의 몫.

**construct-first**: 도구가 아니라 '무엇을, 어떤 증거로, 어디까지'가 먼저입니다.
